# 05 · Ultrack Entegrasyonu

## Neden Ultrack
Klasik detection **0.78'de doygun** (LB 0.749). Denedik ve elendi: parametre taraması (27 konfig),
**watershed** (edge_J 0.34 — felaket), h-maxima, sınır temizleme.
Kök sorun: **temas eden, farklı parlaklıkta çekirdekler** — sönük olan parlak komşusu tarafından
bastırılıyor (local-max olamıyor). Kaçan GT'lerin %67'si lokal olarak sönük.

**Ultrack tam bu problem için tasarlandı:** tek bir segmentasyon seçmek yerine **çoklu aday
hipotez** üretir ve **ILP** ile "hangi segment seti zaman boyunca en tutarlı track'i verir"
sorusunu çözer. Yani ayrım kararını *görüntüden* değil *takip tutarlılığından* alır.

## Kurulum durumu (fizibilite geçti ✅)
- `ultrack 0.7.2` **offline import OK** (internet kapalı, submit ortamı)
- `mip` (COIN-OR CBC) solver var → **Gurobi'siz** çalışır
- Kütüphane `erd-ultrack-libs` dataset'inde tek arşiv olarak

## Bu defterin yaklaşımı: **küçük başla, ölç, sonra ölçekle**
Ultrack'in ILP'si CPU'da yavaş olabilir. Önce **1 dataset × az kare**, süreyi ölç,
çıktı formatını gör, yerel edge_J ile **0.7796**'ya karşı kıyasla. Ancak kazanırsa ölçekle.

## 0 · Offline kurulum

In [ ]:
import subprocess, sys, glob, os, time
LIB="/kaggle/working/ultrack_libs"
if not os.path.isdir(LIB):
    arc=(glob.glob("/kaggle/input/**/ultrack_libs.tgz.bin", recursive=True) or
         glob.glob("/kaggle/input/**/ultrack_libs.tar.gz", recursive=True))
    assert arc, "ultrack arsivi bulunamadi! 'erd-ultrack-libs' dataset'i ekli mi?"
    t0=time.time(); subprocess.run(["tar","-xzf",arc[0],"-C","/kaggle/working"],check=True)
    print(f"arsiv acildi ({time.time()-t0:.0f}s):", arc[0])
# APPEND! insert(0) DEGIL:
#   LIB'de pandas 3.0.3 var; pandas 3.0'da Copy-on-Write default -> np.asarray(series)
#   READ-ONLY doner -> skimage'in Cython _map_array'i patlar
#   ("buffer source array is read-only", ultrack solve asamasinda).
#   append ile ortamin pandas 2.3.3 / numpy 2.0.2 / skimage 0.25.2'si oncelikli olur;
#   ultrack / mip / zarr gibi ortamda OLMAYANLAR LIB'den gelir.
if LIB not in sys.path: sys.path.append(LIB)

import numpy as np, pandas as pd
import ultrack
from ultrack import MainConfig, Tracker
print("ultrack:", ultrack.__version__, "| numpy:", np.__version__, "| pandas:", pd.__version__)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — 3.x Copy-on-Write ultrack'i kirar! sys.path.append kullan.")

# zarr: once ultrack_libs'ten gelir; yoksa cell-tracking-libs
try:
    import zarr
except ImportError:
    for root,dirs,files in os.walk("/kaggle/input"):
        if os.path.basename(root)=="zarr" and "__init__.py" in files:
            sys.path.insert(0, os.path.dirname(root)); break
    import zarr
print("zarr:", zarr.__version__)

from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
import warnings; warnings.filterwarnings("ignore")
SCALE=(1.625,0.40625,0.40625); S=np.array(SCALE,np.float32); MATCH_UM=7.0
print("hazir")

## 1 · API keşfi
Ultrack sürümleri arasında imza değişiyor — **varsayım yapmadan** çalışma anında okuyalım.

In [ ]:
import inspect
def sig(fn, name):
    try: print(f"{name}{inspect.signature(fn)}\n")
    except Exception as e: print(f"{name}: imza okunamadi ({e})\n")

sig(Tracker.track, "Tracker.track")
sig(Tracker.to_tracks_layer, "Tracker.to_tracks_layer")
try:
    from ultrack.imgproc import detect_foreground, robust_invert
    sig(detect_foreground, "detect_foreground"); sig(robust_invert, "robust_invert")
    HAS_IMGPROC=True
except Exception as e:
    print("imgproc yok:", e); HAS_IMGPROC=False
try:
    from ultrack.utils.array import array_apply; sig(array_apply, "array_apply"); HAS_AA=True
except Exception as e:
    print("array_apply yok:", e); HAS_AA=False

cfg=MainConfig()
print("=== MainConfig alanlari ===")
for f in ("data_config","segmentation_config","linking_config","tracking_config"):
    sub=getattr(cfg,f,None)
    if sub is None: continue
    keys=list(sub.model_dump().keys()) if hasattr(sub,"model_dump") else list(vars(sub).keys())
    print(f"{f}: {keys}")

## 2 · Veri

In [ ]:
INPUT=__import__("pathlib").Path("/kaggle/input")
from pathlib import Path
def find_root():
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/"train").is_dir() and (b/"test").is_dir(): return b
        except Exception: pass
        if d<4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr",".geff")): st.append((c,d+1))
ROOT=find_root(); TRAIN=ROOT/"train"; TEST=ROOT/"test"
test_names=sorted(p.stem for p in TEST.glob("*.zarr"))
def open_image(zp):
    n=zarr.open(str(zp),mode="r"); a=dict(n.attrs)
    ms=a.get("multiscales") or (a.get("ome") or {}).get("multiscales")
    if ms: return n[ms[0]["datasets"][0]["path"]]
    return n["0"] if "0" in list(n.keys()) else n
def load_geff(gp):
    g=zarr.open(str(gp),mode="r"); nodes=g["nodes"]; ids=np.asarray(nodes["ids"]); props={}
    for pn in list(nodes["props"].keys()):
        try: props[pn]=np.asarray(nodes["props"][pn]["values"])
        except Exception: pass
    d={"id":ids}
    for k in ("t","z","y","x"):
        if k in props: d[k]=props[k]
    return pd.DataFrame(d), np.asarray(g["edges"]["ids"])

# KUCUK BASLA: en zorlandigimiz dataset, az kare
DS   = "6bba_05b6850b"      # baseline edge_J=0.846, recall=0.92 -> Ultrack asabilir mi?
NT   = 20                    # kare sayisi (once kucuk; sure olcup buyutecegiz)
arr=open_image(TEST/(DS+".zarr"))
img=np.asarray(arr[:NT]).astype(np.float32)
print("dataset:", DS, "| kullanilan blok:", img.shape)
gdf,ge=load_geff(TRAIN/(DS+".geff"))
gdf=gdf[gdf.t<NT]; gtset_all=set((int(u),int(v)) for u,v in ge)
keep=set(gdf.id); ge_sub=np.array([[u,v] for u,v in ge if u in keep and v in keep])
print(f"GT bu blokta: {len(gdf)} node, {len(ge_sub)} edge")

## 3 · Foreground + kontur
Ultrack iki girdi ister: **foreground** (hücre nerede) ve **edges/contours** (sınırlar).
Önce Ultrack'in kendi fonksiyonlarını deneriz; olmazsa bizim Otsu + ters-intensity'ye düşeriz.

In [ ]:
def my_foreground(vol):
    sm=ndi.gaussian_filter(vol, sigma=(1,2,2)); return sm>threshold_otsu(sm)
def my_contours(vol):
    sm=ndi.gaussian_filter(vol, sigma=(1,2,2))
    lo,hi=np.percentile(sm,[1,99.5]); n=np.clip((sm-lo)/max(hi-lo,1e-6),0,1)
    return (1.0-n).astype(np.float32)      # ters intensity = sinir yakinligi

# 1. kosu sonucu: ultrack detect_foreground(sigma=15) fg oranini %5'e daraltti
#   -> GT'lerin ~%25'i foreground DISINDA kaldi -> node_recall 0.746 (baseline 0.971).
#   Ultrack goremedigi hucreyi takip edemez. Bizim Otsu foreground'u GT'lerin %97'sini
#   kapsiyor -> onu besleyip Ultrack'i SADECE instance ayrimi + ILP icin kullaniyoruz.
FG_SOURCE = "kendi"          # "kendi" | "ultrack"

t0=time.time(); USED=FG_SOURCE
if FG_SOURCE=="ultrack":
    try:
        from ultrack.imgproc import detect_foreground, robust_invert
        fg=np.stack([detect_foreground(img[t]) for t in range(NT)]).astype(bool)
        ct=np.stack([robust_invert(img[t]) for t in range(NT)]).astype(np.float32)
    except Exception as e:
        print("[ultrack imgproc basarisiz] ->", type(e).__name__, e); USED="kendi"
if USED=="kendi":
    fg=np.stack([my_foreground(img[t]) for t in range(NT)])
    ct=np.stack([my_contours(img[t]) for t in range(NT)])

# foreground GT'leri kapsiyor mu? (recall TAVANI — bunun ustune cikamaz)
gz=gdf.z.values.astype(int); gy=gdf.y.values.astype(int); gx=gdf.x.values.astype(int); gt_=gdf.t.values.astype(int)
cov=float(np.mean([fg[t,z,y,x] for t,z,y,x in zip(gt_,gz,gy,gx)]))
print(f">> FOREGROUND GT KAPSAMI: {cov:.3f}  (recall TAVANI — 1. kosuda ultrack fg ile ~0.75'ti)")
print(f"foreground+kontur ({USED}) {time.time()-t0:.0f}s | fg orani={fg.mean():.3f} "
      f"| kontur [{ct.min():.2f},{ct.max():.2f}]")
assert fg.any(), "foreground bos!"

import matplotlib.pyplot as plt
t=NT//2
fig,ax=plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(img[t].max(0),cmap="gray"); ax[0].set_title("goruntu (MIP)")
ax[1].imshow(fg[t].max(0),cmap="gray");  ax[1].set_title(f"foreground ({USED})")
ax[2].imshow(ct[t].max(0),cmap="magma"); ax[2].set_title("kontur")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 4 · Config — EDA parametrelerimizle
`scale` verince mesafeler µm olur → `max_distance=8` (EDA 99p). Çekirdek ~10 µm çap:
hacim ≈ 524 µm³, voxel 0.268 µm³ → **~1950 voxel**; min/max alanı buna göre geniş tutuyoruz.

In [ ]:
cfg=MainConfig()
cfg.data_config.working_dir = "/kaggle/working/ultrack_db"
subprocess.run(["rm","-rf","/kaggle/working/ultrack_db"])   # temiz baslangic
os.makedirs("/kaggle/working/ultrack_db", exist_ok=True)

def setattr_safe(obj, name, val):
    if hasattr(obj,name):
        try: setattr(obj,name,val); return True
        except Exception as e: print(f"  [!] {name} set edilemedi: {e}")
    else: print(f"  [!] alan yok: {name}")
    return False

print("segmentation:")
setattr_safe(cfg.segmentation_config,"min_area",300)      # cok kucuk parcalari at
setattr_safe(cfg.segmentation_config,"max_area",8000)     # birlesmis kumeleri at
setattr_safe(cfg.segmentation_config,"n_workers",2)
print("linking:")
setattr_safe(cfg.linking_config,"max_distance",8.0)       # EDA 99p (um, scale ile)
setattr_safe(cfg.linking_config,"max_neighbors",5)
setattr_safe(cfg.linking_config,"n_workers",2)
print("tracking:")
setattr_safe(cfg.tracking_config,"division_weight",-0.01) # bolunmeye izin (nadir)
setattr_safe(cfg.tracking_config,"appear_weight",-0.001)
setattr_safe(cfg.tracking_config,"disappear_weight",-0.001)
setattr_safe(cfg.tracking_config,"n_threads",2)
print("\nKONFIG:", cfg)

## 5 · Ultrack çalıştır — **süreyi ölç** (asıl risk burada)
ILP CPU'da yavaş olabilir. Bu blok 20 kare; 100 kare × N dataset'e sığar mı ekstrapole edeceğiz.

In [ ]:
tracker=Tracker(cfg)
t0=time.time(); ok=True
try:
    try:
        tracker.track(foreground=fg, edges=ct, scale=SCALE)     # olcek destekleniyorsa
        print("track(scale=...) ile calisti")
    except TypeError:
        tracker.track(foreground=fg, edges=ct)                  # eski imza
        print("track() (scale'siz) ile calisti — mesafeler VOXEL")
except Exception as e:
    ok=False; print("TRACK PATLADI:", type(e).__name__, e)
    import traceback; traceback.print_exc()
el=time.time()-t0
print(f"\nsure: {el:.0f}s ({NT} kare)")
if ok:
    print(f"tahmini 1 dataset (100 kare): {el*100/NT/60:.1f} dk")
    print(f"tahmini 4 dataset: {el*100/NT*4/60:.1f} dk")

## 6 · Sonuçları al → bizim graf formatımıza çevir

In [ ]:
assert ok, "track() patladi -> Bolum 5 hatasini duzelt, sonra buraya don"
tracks_df, graph = tracker.to_tracks_layer()
print("tracks_df kolonlari:", list(tracks_df.columns))
print(tracks_df.head())
print("track sayisi:", tracks_df.track_id.nunique(), "| node:", len(tracks_df))
print("graph (bolunme) ornek:", dict(list(graph.items())[:5]) if graph else "yok")

# napari tracks -> node/edge grafi
cols=list(tracks_df.columns)
zc = "z" if "z" in cols else None
tr=tracks_df.sort_values(["track_id","t"]).reset_index(drop=True)
tr["node_id"]=np.arange(1,len(tr)+1)
nodes=[(int(r.node_id),int(r.t),float(getattr(r,"z",0.0)),float(r.y),float(r.x)) for r in tr.itertuples()]
edges=[]
last_of={}; first_of={}
for tid,g in tr.groupby("track_id"):
    g=g.sort_values("t"); ids=g.node_id.values
    edges += [(int(ids[i]),int(ids[i+1])) for i in range(len(ids)-1)]   # track ici
    first_of[tid]=int(ids[0]); last_of[tid]=int(ids[-1])
for child,parents in (graph or {}).items():                             # bolunme kenarlari
    for p in (parents if isinstance(parents,(list,tuple)) else [parents]):
        if p in last_of and child in first_of: edges.append((last_of[p], first_of[child]))
print(f"\nbizim graf: {len(nodes)} node, {len(edges)} edge")

## 7 · Yerel metrik — baseline'a karşı
Aynı blok (ilk {NT} kare) için baseline'ı da koşup **aynı zeminde** kıyaslıyoruz.

In [ ]:
def eval_edges(nodes, edges, gt_ndf, gt_edges):
    pn=pd.DataFrame(nodes,columns=["node_id","t","z","y","x"]); gmap={}
    for t,g in gt_ndf.groupby("t"):
        p=pn[pn.t==int(t)]
        if len(p)==0 or len(g)==0: continue
        D=cdist(p[["z","y","x"]].values*S, g[["z","y","x"]].values*S)
        cost=np.where(D<=MATCH_UM,D,1e6); r,c=linear_sum_assignment(cost)
        pid=p.node_id.values; gid=g.id.values
        for i,j in zip(r,c):
            if D[i,j]<=MATCH_UM: gmap[int(pid[i])]=int(gid[j])
    gset=set((int(u),int(v)) for u,v in gt_edges); TP=0;FP=0;cov=set()
    for u,v in edges:
        gu=gmap.get(u); gv=gmap.get(v)
        if gu is None or gv is None: continue
        if (gu,gv) in gset: TP+=1; cov.add((gu,gv))
        else: FP+=1
    FN=len(gset)-len(cov)
    return dict(edge_J=round(TP/max(TP+FP+FN,1),4), TP=TP, FP=FP, FN=FN,
                node_recall=round(len(set(gmap.values()))/max(len(gt_ndf),1),3), nodes=len(nodes))

# --- BASELINE ayni blokta ---
def base_detect(v):
    sm=ndi.gaussian_filter(v,sigma=(1,2,2)); thr=threshold_otsu(sm)
    mx=ndi.maximum_filter(sm,size=(3,11,11)); pk=(sm==mx)&(sm>thr)
    lbl,n=ndi.label(pk)
    if n==0: return np.zeros((0,3),np.float32)
    return np.asarray(ndi.center_of_mass(sm,lbl,np.arange(1,n+1)),np.float32)
cents=[base_detect(img[t]) for t in range(NT)]
bn=[];be=[];off=[];nid=1
for t,c in enumerate(cents):
    off.append(nid)
    for p in c: bn.append((nid,t,float(p[0]),float(p[1]),float(p[2]))); nid+=1
for t in range(NT-1):
    A,B=cents[t],cents[t+1]
    if len(A) and len(B):
        D=cdist(A*S,B*S); r,c=linear_sum_assignment(np.where(D<=8.0,D,1e6))
        be+=[(off[t]+int(i),off[t+1]+int(j)) for i,j in zip(r,c) if D[i,j]<=8.0]

print("=== ULTRACK  ===", eval_edges(nodes,edges,gdf,ge_sub))
print("=== BASELINE ===", eval_edges(bn,be,gdf,ge_sub))
print("\n(tam dataset baseline referansi: edge_J=0.846 bu dataset icin, micro 0.7796 genel)")

## 8 · Karar

- [ ] Ultrack edge_J **>** baseline mı (aynı blokta)?
- [ ] Süre: 1 dataset kaç dk → gizli test setine sığar mı?
- [ ] `node_recall` yükseldi mi (asıl hedef: temas eden çekirdekleri ayırmak)?

**Kazanırsa:** tüm dataset'lere ölçekle → `02_baseline`'ın detection+linking'ini Ultrack ile değiştir → submit.
**Kaybederse:** config ayarı (min_area/max_area, contour kaynağı) → yine olmazsa Plan C (track budama).